# 02 · Data pipeline

Replaces `scripts/ingest.py` (plan Section 4):

1. **Scrape** Facebook Marketplace with the Apify actor (`startUrls`, `resultsLimit`, a spend
   cap). Raw results go to `data/raw/` (gitignored: seller names and URLs).
2. **Anonymize**: drop seller name, seller id and URL; keep city. Ids are a hash of the
   Facebook id, so they stay the same on every scrape and the hand labels keep matching.
3. **Identify** the card with the LLM, 20 listings per call, up to 4 calls at a time. A rule
   pre-filter first drops what the title already shows is out of scope (graded slabs, lots,
   sealed packs, non-English; in a live search also listings that never name the card), and
   cards identified before are reused from `.cache/identified`, so they cost no LLM call.
4. **Scope filter**: drop graded cards, lots, non-English printings, unidentified cards.
5. **Prices**: resolve set names to TCGdex ids, fetch TCGplayer market prices once, cache
   them in `data/prices.json`.

Output: `data/listings.json`, `data/prices.json`, and `data/listings_review.csv` for the
hand check, with a "why" for each row that needs one. The same functions serve the Merchant
Agent's live search.

Added after the first real scrape (100 listings, 2026-11-25):

- **$1 listings** are "make an offer" placeholders, like $0, and are dropped.
- **Listing signals** for the score: photo count, posting date, distance from the buyer's
  city, whether a public meetup is offered, pending. Only the distance is kept, not the
  item's coordinates.
- **The printed set size decides the set**: "231/182" can only be a 182-card set. When
  several sets share the size, or the card name doesn't match the card TCGdex has at that
  number ("Sabrina's Gengar" at Gym Challenge 14 is Rocket's Mewtwo), each candidate set is
  checked on TCGdex and the one with this card wins. If none has it, the listing gets no
  market price and is flagged for the hand check.
- **Set-name aliases** in `tcgdex.set_aliases` ("Scarlet & Violet Black Star Promos" →
  `svp`), and promo numbers ("XY121") found through the set's card list.

Fixed from `ingest.py`: the actor input (`searchQuery`/`maxItems` → `startUrls`/`resultsLimit`),
apify-client 3.x returning a Run object, no spend cap, descriptions saved as `"{'text': ...}"`,
row-number ids, missing prices becoming $0, one LLM call per listing, and TCGdex lookups
with raw model output ("4/102", set names) instead of resolved set ids.

In [ ]:
#| default_exp pipeline

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Data pipeline (Section 4): Facebook Marketplace listings through Apify -> anonymize ->
LLM card identification -> scope filter -> TCGdex market prices. Also loads the listing pool
the Merchant Agent searches, and serves its optional live search."""

import csv
import hashlib
import json
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from decimal import Decimal
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus

import httpx
from diskcache import Cache
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

from src.config import env, repo_path
from src.llm import LLM, make_llm, offline_handler
from src.schemas import CardListing, Condition, Variant
from src.tools.risk import RedFlag, parse_red_flags

## 1. Scrape

In [ ]:
#| export
class SourceError(RuntimeError):
    pass

In [ ]:
#| export
def search_url(city: str, term: str) -> str:
    return f"https://www.facebook.com/marketplace/{city}/search/?query={quote_plus(term)}"

In [ ]:
#| export
class MarketplaceScraper:
    """Runs the Apify Facebook Marketplace actor. Pass `client` to use a fake in tests."""

    def __init__(self, config: dict[str, Any], client=None):
        self.cfg = config
        self._client = client

    @property
    def client(self):
        if self._client is None:
            token = env("APIFY_API_TOKEN", "APIFY_TOKEN")
            if not token:
                raise SourceError("APIFY_API_TOKEN is missing. Add it to .env (see .env.example).")
            from apify_client import ApifyClient

            self._client = ApifyClient(token)
        return self._client

    def search(
        self,
        terms: list[str],
        limit: int,
        max_charge_usd: float,
        details: bool = True,
        city: str | None = None,
        save_as: str | None = "snapshot",
    ) -> list[dict]:
        """Scrape the search terms (one Apify run) and return the raw items. Unless save_as is
        None, they are also saved to data/raw/, which is gitignored (it has listing links)."""
        s = self.cfg["scrape"]
        run_input = {
            "startUrls": [{"url": search_url(city or s["city"], t)} for t in terms],
            "resultsLimit": limit,
            "includeListingDetails": details,
        }
        try:
            run = self.client.actor(s["actor"]).call(
                run_input=run_input, max_total_charge_usd=Decimal(str(max_charge_usd))
            )
        except SourceError:
            raise
        except Exception as e:
            raise SourceError(f"Apify run failed: {e}") from e
        if run is None:
            raise SourceError("Apify run did not start. Check the token and credit in the console.")
        # apify-client 3.x returns a Run object; older versions returned a dict.
        dataset_id = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
        items = list(self.client.dataset(dataset_id).iterate_items())
        if save_as:
            save_raw(self.cfg, items, save_as)
        return items

In [ ]:
#| export
class ReplayScraper:
    """Stands in for MarketplaceScraper with a saved raw scrape (data/raw/*.json): the live
    search runs end to end for free, on the same listings every time."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.items = json.loads(self.path.read_text(encoding="utf-8"))

    def search(self, terms: list[str], limit: int, max_charge_usd: float, **_ignored) -> list[dict]:
        return self.items[:limit]

In [ ]:
#| export
def save_raw(config: dict[str, Any], items: list[dict], prefix: str) -> Path:
    out_dir = repo_path(config["data"]["raw_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}.json"
    path.write_text(json.dumps(items, indent=1, ensure_ascii=False), encoding="utf-8")
    return path

In [ ]:
#| export
def scrape_snapshot(
    config: dict[str, Any], scraper: MarketplaceScraper, per_query: int | None = None, log=print
) -> list[dict]:
    """The snapshot: one Apify run per search in `scrape.queries`, so every search
    contributes `per_query` listings. A listing found by two searches is kept once. The spend
    cap `scrape.max_charge_usd` is split evenly across the runs. Saves one combined raw file.
    """
    s = config["scrape"]
    n = per_query or s["per_query"]
    cap = s["max_charge_usd"] / len(s["queries"])
    seen: set[str] = set()
    items: list[dict] = []
    for q in s["queries"]:
        got = scraper.search([q], n, cap, details=s["include_listing_details"], save_as=None)
        new = [i for i in got if str(first(i, "id", "listing_id")) not in seen]
        seen.update(str(first(i, "id", "listing_id")) for i in new)
        items += new
        if log:
            log(f"  {q!r}: {len(got)} listings, {len(new)} new")
    path = save_raw(config, items, "snapshot")
    if log:
        log(f"{len(items)} listings saved to {path.relative_to(repo_path('.'))}")
    return items

In [ ]:
#| export
def latest_raw(config: dict[str, Any], prefix: str = "snapshot") -> Path | None:
    """Newest raw file in data/raw/ (to redo extraction without scraping again)."""
    files = sorted(repo_path(config["data"]["raw_dir"]).glob(f"{prefix}_*.json"))
    return files[-1] if files else None

## 2. Anonymize

In [ ]:
#| export
def first(item: dict, *paths: str):
    """First non-empty value among dotted paths, e.g. 'location.reverse_geocode.city'."""
    for p in paths:
        cur: Any = item
        for part in p.split("."):
            cur = cur.get(part) if isinstance(cur, dict) else None
            if cur is None:
                break
        if cur not in (None, ""):
            return cur
    return None

In [ ]:
#| export
def to_price(value) -> float | None:
    if isinstance(value, dict):
        value = value.get("amount") or value.get("formatted_amount")
    if isinstance(value, int | float):
        return float(value)
    if isinstance(value, str):
        m = re.search(r"\d[\d,]*(?:\.\d+)?", value)
        return float(m.group(0).replace(",", "")) if m else None
    return None

In [ ]:
#| export
def anon_id(source_id: str) -> str:
    """Stable anonymous id: the same listing gets the same id on every scrape, so hand
    labels and the answer key stay valid, but the Facebook id itself is not stored."""
    return "fb_" + hashlib.sha256(str(source_id).encode()).hexdigest()[:10]

In [ ]:
#| export
def delivery_type(types: Any) -> str:
    text = " ".join(types).lower() if isinstance(types, list) else str(types or "").lower()
    if any(w in text for w in ("in_person", "pickup", "local")):
        return "in_person"  # shipping is $0 when pickup is offered
    if "ship" in text:
        return "shipping"
    return "unknown"

In [ ]:
#| export
def miles(a: tuple[float, float], b: tuple[float, float]) -> float:
    """Great-circle distance between two (lat, lon) points, in miles."""
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    )
    return 3958.8 * 2 * math.asin(math.sqrt(h))

In [ ]:
#| export
def anonymize(item: dict, config: dict[str, Any] | None = None) -> dict | None:
    """Map one raw Apify item to the fields we keep (Section 14). Seller details and the
    listing URL are dropped; location is kept at city level, plus the distance from the
    buyer's city (config `buyer`). Returns None for sold or hidden listings and items without
    a title, id or a real price: $0 or, with a config, up to `ingest.placeholder_max_price`
    ($1 "make an offer" placeholders).

    Field names are the ones the actor returned in the first real scrape (2026-11-24):
    listingTitle, listingPrice.amount, description.text, condition, deliveryTypes,
    locationText.text, location.latitude, listingPhotos, timestamp, isPending, isSold. Older
    snake_case names still work.
    """
    if first(item, "isSold", "is_sold") is True or first(item, "isHidden") is True:
        return None
    title = first(item, "listingTitle", "marketplace_listing_title", "custom_title", "title")
    price = to_price(first(item, "listingPrice", "listing_price", "price"))
    source_id = first(item, "id", "listing_id")
    floor = float(config["ingest"]["placeholder_max_price"]) if config else 0.0
    if not (title and source_id and price) or price <= floor:
        return None
    desc = first(item, "description.text", "redacted_description.text", "description")
    photos = first(item, "listingPhotos", "listing_photos", "photos")
    primary = first(item, "primaryListingPhoto", "primary_listing_photo")
    n_photos = len(photos) if isinstance(photos, list) else int(bool(primary))
    types = first(item, "deliveryTypes", "delivery_types") or []
    lat, lon = first(item, "location.latitude"), first(item, "location.longitude")
    distance = None
    if config and isinstance(lat, int | float) and isinstance(lon, int | float):
        b = config["buyer"]
        distance = round(miles((lat, lon), (b["lat"], b["lon"])), 1)
    posted = first(item, "timestamp", "creation_time")
    return {
        "listing_id": anon_id(source_id),
        "raw_title": str(title).strip(),
        "raw_description": desc.strip() if isinstance(desc, str) else "",
        "price": price,
        "location_city": first(item, "locationText.text", "location.reverse_geocode.city") or "",
        "delivery_type": delivery_type(first(item, "deliveryTypes", "delivery_types")),
        "has_clear_photo": n_photos > 0,
        "photo_count": n_photos,
        "listed_at": str(posted)[:10] if isinstance(posted, str) and posted[:4].isdigit() else None,
        "distance_miles": distance,
        "public_meetup": isinstance(types, list) and "PUBLIC_MEETUP" in types,
        "pending": first(item, "isPending", "is_pending") is True,
        # Facebook's generic label ("Used - like new"): only a hint for card identification,
        # not stored in the listing.
        "condition_label": first(item, "condition") or "",
    }

## 3. Identify the card, then filter the scope

In [ ]:
#| export
class ExtractedCard(BaseModel):
    listing_id: str
    card_name: str | None = None
    set_name: str | None = None
    number: str | None = None
    variant: Variant | None = None
    condition: Condition = "unknown"
    graded: bool = False
    grade: str | None = None
    language: str = "en"
    is_lot: bool = False
    shipping: float | None = Field(default=None, ge=0.0)
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)

In [ ]:
#| export
class ExtractionBatch(BaseModel):
    cards: list[ExtractedCard]

In [ ]:
#| export
EXTRACT_SYSTEM = """You identify the exact Pokémon trading card in each marketplace listing.
For every listing return:
- card_name: English card name with suffix (e.g. "Charizard", "Charizard ex", "Pikachu VMAX").
- set_name: the set, from the text or implied by the number (4/102 -> "Base Set",
  199/165 -> "151", 44/185 -> "Vivid Voltage"). Null if you can't tell.
- number: the number before the slash, without leading zeros ("044/185" -> "44").
- variant: normal, holo, reverse or 1st edition, if stated.
- condition: NM, LP, MP, HP, DMG, or unknown. "Mint"/"pack fresh" -> NM, "light wear" -> LP,
  "well loved"/"heavily played" -> HP, creases, bends or water damage -> DMG. Only if the title
  and description say nothing about condition, use facebook_condition (the seller's generic
  label) as a weak hint: "New" -> NM, "Used - like new" -> LP, "Used - good" -> MP,
  "Used - fair" -> HP.
- graded/grade: true and the grade if the card is in a PSA, BGS or CGC slab.
- language: two-letter code; "en" unless another language is stated.
- is_lot: true for bundles, lots, binders, boxes, sealed product, or several different cards.
- shipping: shipping cost in USD only if the text states it. Never assume $0.
- confidence: 0-1 that card_name, set_name and number are all correct.
Return one entry per listing, with the same listing_id."""

In [ ]:
#| export
class IdentifiedCache:
    """Cards the LLM has already identified, keyed by model and listing text, so a listing seen
    in an earlier search (or an earlier ingest) isn't sent to the LLM again. On disk at
    `ingest.identified_cache`; None turns it off. Changed text means a new key."""

    def __init__(self, config: dict[str, Any]):
        path = config["ingest"].get("identified_cache")
        self.store = Cache(str(repo_path(path))) if path else None

    @staticmethod
    def key(model: str, row: dict) -> str:
        blob = json.dumps([model, row["raw_title"], row["raw_description"],
                           row.get("condition_label", "")], ensure_ascii=False)  # fmt: skip
        return hashlib.sha256(blob.encode()).hexdigest()

    def get(self, model: str, row: dict) -> dict | None:
        return None if self.store is None else self.store.get(self.key(model, row))

    def put(self, model: str, row: dict, card: dict) -> None:
        if self.store is not None:
            self.store[self.key(model, row)] = card

In [ ]:
#| export
def extract_cards(
    llm: LLM,
    rows: list[dict],
    batch_size: int,
    log=print,
    parallel: int = 1,
    cache: IdentifiedCache | None = None,
    stats: dict | None = None,
) -> dict[str, dict]:
    """Identify the card in each row, `batch_size` listings per LLM call and up to `parallel`
    calls at a time. Rows found in `cache` skip the LLM. `stats` gets the counts."""
    out: dict[str, dict] = {}
    todo = []
    for r in rows:
        hit = cache.get(llm.model, r) if cache else None
        if hit is not None:
            out[r["listing_id"]] = {**hit, "listing_id": r["listing_id"]}
        else:
            todo.append(r)
    batches = [todo[i : i + batch_size] for i in range(0, len(todo), batch_size)]

    def identify(batch: list[dict]) -> dict[str, dict]:
        items = [{"listing_id": r["listing_id"], "title": r["raw_title"],
                  "description": r["raw_description"],
                  "facebook_condition": r.get("condition_label", "")} for r in batch]  # fmt: skip
        prompt = "Listings (JSON):\n<listings>\n" + json.dumps(items, ensure_ascii=False)
        prompt += "\n</listings>"
        res = llm.json(tag="extract", system=EXTRACT_SYSTEM, prompt=prompt, schema=ExtractionBatch)
        return {c.listing_id: c.model_dump() for c in res.cards}

    done = len(out)
    with ThreadPoolExecutor(max(1, parallel)) as pool:
        for batch, found in zip(batches, pool.map(identify, batches), strict=True):
            out.update(found)
            for r in batch:
                if r["listing_id"] in found and cache:
                    cache.put(llm.model, r, found[r["listing_id"]])
            done += len(batch)
            if log:
                log(f"  identified cards in {done}/{len(rows)} listings")
    if stats is not None:
        stats.update({"listings": len(rows), "from_cache": len(rows) - len(todo),
                      "llm_calls": len(batches), "parallel": max(1, parallel)})  # fmt: skip
    return out

In [ ]:
#| export
def build_listings(rows: list[dict], extracted: dict[str, dict]) -> list[CardListing]:
    """Merge anonymized rows with the extracted fields. The row's own fields (id, price)
    always win, so the model can never change a price."""
    listings = []
    for r in rows:
        card = {k: v for k, v in extracted.get(r["listing_id"], {}).items() if v is not None}
        card.pop("listing_id", None)
        listings.append(CardListing.model_validate({**card, **r}))
    return listings

In [ ]:
#| export
def scope_reason(card: CardListing) -> str | None:
    """Why a listing is out of scope (Section 4, step 3), or None if it is in scope."""
    if card.graded:
        return "graded"
    if card.is_lot:
        return "is_lot"
    if card.language.lower() != "en":
        return "non_english"
    if card.card_name == "unknown":
        return "unidentified"
    return None

In [ ]:
#| export
def apply_scope_filter(listings: list[CardListing]) -> tuple[list[CardListing], dict[str, int]]:
    """Drop graded cards, lots/binders, non-English printings and unidentified cards."""
    kept: list[CardListing] = []
    dropped = {"graded": 0, "is_lot": 0, "non_english": 0, "unidentified": 0}
    for card in listings:
        reason = scope_reason(card)
        if reason:
            dropped[reason] += 1
        else:
            kept.append(card)
    return kept, dropped

In [ ]:
#| export
GRADED_TITLE = re.compile(
    r"\b(psa|bgs|cgc|sgc)\s*-?\s*(10|[1-9](\.5)?)\b(?!\s*(worthy|candidate|ready|potential))|"
    r"\bslab(bed)?\b",
    re.IGNORECASE,
)
LOT_TITLE = re.compile(
    r"\b(lots?|bundle|binder|bulk|booster\s+(box|pack|bundle)s?|elite\s+trainer\s+box|etb|"
    r"sealed\s+(box|pack|product|case)|\d{2,}\s*(cards|pcs))\b",
    re.IGNORECASE,
)
FOREIGN_TITLE = re.compile(
    r"\b(japanese|korean|chinese|french|german|spanish|italian|portuguese)\b", re.IGNORECASE
)
NOT_A_NAME = {"ex", "gx", "v", "vmax", "vstar", "lv", "x", "m", "mega", "team", "s", "the", "of"}

In [ ]:
#| export
def prefilter(rows: list[dict], card_name: str | None = None) -> tuple[list[dict], dict[str, int]]:
    """Drop what the title already shows is out of scope, before any LLM call: graded slabs
    ("PSA 9", not "PSA 10 worthy"), lots and sealed product, non-English cards. For a live
    search (`card_name`), also listings whose title and description never name the card
    (fuzzy, so "Charzard" still counts). Deliberately cautious: anything unclear goes on to
    the LLM, and the scope filter after it."""
    names = [t for t in re.findall(r"[a-z0-9]+", (card_name or "").lower())
             if t not in NOT_A_NAME and len(t) >= 3]  # fmt: skip
    kept, dropped = [], {}
    for r in rows:
        title, text = r["raw_title"], f"{r['raw_title']} {r['raw_description']}".lower()
        if GRADED_TITLE.search(title):
            reason = "graded"
        elif LOT_TITLE.search(title):
            reason = "is_lot"
        elif FOREIGN_TITLE.search(title):
            reason = "non_english"
        elif names and not any(
            fuzz.ratio(n, w) >= 80 for n in names for w in re.findall(r"[a-z0-9]+", text)
        ):
            reason = "other_card"
        else:
            kept.append(r)
            continue
        dropped[reason] = dropped.get(reason, 0) + 1
    return kept, dropped

In [ ]:
#| export
def ingest_llm(config: dict[str, Any], default: LLM) -> LLM:
    """The model that identifies cards: `ingest.llm` (a key under `models:`), else `default`."""
    key = config["ingest"].get("llm")
    return make_llm(config, key) if key else default

In [ ]:
#| export
def identify_listings(
    config: dict[str, Any], llm: LLM, rows: list[dict], log=print, card_name: str | None = None
) -> tuple[list[CardListing], dict[str, int], dict[str, Any]]:
    """Anonymized rows -> (in-scope listings, dropped counts by reason, stats). The title
    pre-filter (`ingest.prefilter`) drops the obvious cases without an LLM call; the rest is
    identified in parallel batches (`ingest.parallel_calls`), reusing cards identified before
    (`ingest.identified_cache`), then the scope filter runs."""
    ing = config["ingest"]
    pre: dict[str, int] = {}
    if ing.get("prefilter", True):
        rows, pre = prefilter(rows, card_name)
    model = ingest_llm(config, llm)
    stats: dict[str, Any] = {"model": model.model, "prefiltered": pre}
    cache = IdentifiedCache(config) if model.provider != "offline" else None
    extracted = extract_cards(model, rows, ing["batch_size"], log=log,
                              parallel=int(ing.get("parallel_calls", 1)), cache=cache,
                              stats=stats)  # fmt: skip
    kept, dropped = apply_scope_filter(build_listings(rows, extracted))
    for reason, n in pre.items():
        dropped[reason] = dropped.get(reason, 0) + n
    return kept, dropped, stats

In [ ]:
#| export
def review_reasons(
    listings: list[CardListing], prices: dict[str, dict], mismatched: list[str], min_conf: float
) -> dict[str, str]:
    """listing_id -> why it needs a hand check: low confidence, no market price, or a name
    that differs from the card TCGdex has at that set and number."""
    out = {}
    for x in listings:
        why = []
        if x.listing_id in mismatched:
            why.append("name differs from TCGdex")
        if x.confidence < min_conf:
            why.append("low confidence")
        if not (x.card_key and x.card_key in prices):
            why.append("no market price")
        if why:
            out[x.listing_id] = "; ".join(why)
    return out

In [ ]:
#| export
def write_review_sheet(
    listings: list[CardListing], config: dict[str, Any], reasons: dict[str, str] | None = None
) -> Path:
    """CSV for the hand check: fill in checked_ok, fix wrong rows in data/listings.json."""
    reasons = reasons or {}
    out = repo_path(config["data"]["review_sheet"])
    cols = ["listing_id", "raw_title", "price", "card_name", "set_name", "set_code", "number",
            "variant", "condition", "confidence"]  # fmt: skip
    with open(out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([*cols, "needs_review", "why", "checked_ok"])
        for x in listings:
            why = reasons.get(x.listing_id, "")
            w.writerow([getattr(x, c) for c in cols] + [bool(why), why, ""])
    return out

## 4. Market prices

In [ ]:
#| export
class TCGdex:
    """Free TCGdex API (no key). Prices come from TCGplayer through TCGdex."""

    def __init__(self, base_url: str, timeout_s: float = 15):
        self.base = base_url.rstrip("/")
        self.http = httpx.Client(timeout=timeout_s, follow_redirects=True)
        self._set_cards: dict[str, list[dict]] = {}

    def _get(self, url: str):
        r = self.http.get(url)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()

    def sets(self) -> list[dict]:
        return self._get(f"{self.base}/sets") or []

    def card(self, set_id: str, number: str) -> dict | None:
        """The card at this set and number. Promo sets number their cards "XY121" or
        "SWSH050"; if the plain number isn't found, the set's own card list is searched."""
        for local in dict.fromkeys([number, number.lstrip("0") or "0", number.zfill(3)]):
            data = self._get(f"{self.base}/cards/{set_id}-{local}")
            if data:
                return data
        digits = re.sub(r"\D", "", number).lstrip("0")
        for c in self.set_cards(set_id):
            if digits and re.sub(r"\D", "", str(c.get("localId", ""))).lstrip("0") == digits:
                return self._get(f"{self.base}/cards/{c['id']}")
        return None

    def set_cards(self, set_id: str) -> list[dict]:
        if set_id not in self._set_cards:
            data = self._get(f"{self.base}/sets/{set_id}") or {}
            self._set_cards[set_id] = data.get("cards") or []
        return self._set_cards[set_id]

    @staticmethod
    def price_entry(card: dict) -> dict:
        """Flatten a TCGdex card into the prices.json entry format."""
        tcg = (card.get("pricing") or {}).get("tcgplayer") or {}
        variants = {
            k: (v.get("marketPrice") or v.get("midPrice"))
            for k, v in tcg.items()
            if isinstance(v, dict)
        }
        return {
            "name": card.get("name"),
            "set_name": (card.get("set") or {}).get("name"),
            "variants": {k: v for k, v in variants.items() if v},
            "currency": tcg.get("unit", "USD"),
            "updated": tcg.get("updated"),
            "source": "tcgdex/tcgplayer",
        }

In [ ]:
#| export
SERIES_PREFIX = re.compile(
    r"^(scarlet\s*&\s*violet|sword\s*&\s*shield|sun\s*&\s*moon|xy|black\s*&\s*white|sv|swsh)\W+",
    re.IGNORECASE,
)

In [ ]:
#| export
def resolve_set(name: str, sets: list[dict], aliases: dict[str, str] | None = None) -> str | None:
    """Map a set name to a TCGdex set id: an alias from config (`tcgdex.set_aliases`) or an
    exact match first, then a close fuzzy match.

    Exact-first matters: 'Base Set' must not resolve to 'Base Set 2'. Series prefixes are
    stripped, so 'Scarlet & Violet—151' finds '151'. A name that already is an id passes.
    """
    alias = {k.casefold(): v for k, v in (aliases or {}).items()}.get(name.casefold().strip())
    if alias:
        return alias
    ids = {s["id"].casefold(): s["id"] for s in sets if s.get("id")}
    if name.casefold() in ids:
        return ids[name.casefold()]
    by_name = {s["name"].casefold(): s["id"] for s in sets if s.get("name")}
    tries = [name, re.split(r"\s*[—–]\s*|\s+-\s+", name)[-1], SERIES_PREFIX.sub("", name)]
    for t in dict.fromkeys(x.strip() for x in tries if x and x.strip()):
        if t.casefold() in by_name:
            return by_name[t.casefold()]
    hit = process.extractOne(name.casefold(), list(by_name), scorer=fuzz.ratio, score_cutoff=88)
    return by_name[hit[0]] if hit else None

In [ ]:
#| export
PRINTED_NUMBER = re.compile(r"\b\d{1,3}\s*/\s*(\d{2,3})\b")

In [ ]:
#| export
def printed_total(listing: CardListing) -> str | None:
    """The set size printed after the card number ("231/182" -> "182"), if the text has one."""
    m = PRINTED_NUMBER.search(f"{listing.raw_title} {listing.raw_description}")
    return m.group(1).lstrip("0") if m else None

In [ ]:
#| export
def is_pocket_set(s: dict) -> bool:
    """TCG Pocket (the phone game) sets are in TCGdex's list too; they are never on sale."""
    return "/tcgp/" in f"{s.get('logo', '')}{s.get('symbol', '')}"

In [ ]:
#| export
def card_names_match(ours: str, theirs: str) -> bool:
    """Same card despite spelling: "Charizard-EX" ~ "Charizard EX", "M Gengar EX" ~ "Mega
    Gengar ex", "Charizard" ~ "Charizard ex". "M Gengar EX" vs "Purrloin" does not match."""

    def norm(n: str) -> str:
        n = re.sub(r"[^a-z0-9' ]", " ", n.lower().replace("-", " "))
        return re.sub(r"\bmega\b", "m", n)

    return fuzz.token_set_ratio(norm(ours), norm(theirs)) >= 80

In [ ]:
#| export
class PriceBook:
    """Market prices for listings: resolves set ids and fetches missing cards from TCGdex.
    Prices are cached in data/prices.json and the set list in data/tcgdex_sets.json, so each
    card is fetched only once."""

    def __init__(
        self, config: dict[str, Any], prices: dict[str, dict], client: TCGdex | None = None
    ):
        self.prices = prices
        self.client = client or TCGdex(config["tcgdex"]["base_url"], config["tcgdex"]["timeout_s"])
        self.prices_path = repo_path(config["data"]["prices"])
        self.sets_path = repo_path(config["data"]["tcgdex_sets"])
        self.aliases = config["tcgdex"].get("set_aliases") or {}
        self.mismatch_conf = float(config["ingest"]["name_mismatch_confidence"])
        self._sets: list[dict] | None = None

    def sets(self) -> list[dict]:
        if self._sets is None:
            if self.sets_path.exists():
                self._sets = json.loads(self.sets_path.read_text(encoding="utf-8"))
            else:
                try:
                    self._sets = self.client.sets()
                except httpx.HTTPError as e:
                    raise SourceError(f"TCGdex set list unavailable: {e}") from e
                self.sets_path.parent.mkdir(parents=True, exist_ok=True)
                self.sets_path.write_text(json.dumps(self._sets, indent=1), encoding="utf-8")
        return self._sets

    def sets_by_size(self) -> dict[str, list[str]]:
        """Printed set size -> TCGdex set ids, e.g. "182" -> ["sv04", "sv10"]."""
        out: dict[str, list[str]] = {}
        for s in self.sets():
            total = str((s.get("cardCount") or {}).get("official") or "")
            if total and total != "0" and not is_pocket_set(s):
                out.setdefault(total, []).append(s["id"])
        return out

    def resolve_sets(self, listings: list[CardListing]) -> list[str]:
        """Fill set_code from set_name; if the printed set size ("231/182") fits exactly one
        set, that set wins over the LLM's guess. Returns the set names that could not be
        matched. Sizes shared by several sets are settled by verify_cards()."""
        by_size = self.sets_by_size()
        unresolved = set()
        for x in listings:
            if x.set_code == "unknown" and x.set_name:
                x.set_code = resolve_set(x.set_name, self.sets(), self.aliases) or "unknown"
            ids = by_size.get(printed_total(x) or "", [])
            if len(ids) == 1 and x.set_code != ids[0]:
                x.set_code = ids[0]
            if x.set_code == "unknown" and x.set_name:
                unresolved.add(x.set_name)
        return sorted(unresolved)

    def verify_cards(self, listings: list[CardListing]) -> tuple[list[str], list[str]]:
        """Check each card against TCGdex. Run after ensure_prices().

        A listing whose name differs from the card TCGdex has at its set and number (e.g.
        "Sabrina's Gengar" at gym2-14, which is Rocket's Mewtwo), or whose set is unknown,
        is tried in every set with its printed size: the set that has this card at this
        number wins (Sabrina's Gengar is gym1-14). If none does, its set is cleared, so it
        gets no market price, and its confidence is capped for the hand check.
        Returns (fixed ids, mismatched ids)."""
        by_size = self.sets_by_size()
        fixed, bad = [], []
        for x in listings:
            entry = self.prices.get(x.card_key or "")
            named_wrong = bool(entry and entry.get("name")) and not card_names_match(
                x.card_name, entry["name"]
            )
            unknown_set = x.set_code == "unknown" and x.number != "unknown"
            if not (named_wrong or unknown_set):
                continue
            for set_id in by_size.get(printed_total(x) or "", []):
                if set_id == x.set_code:
                    continue
                try:
                    card = self.client.card(set_id, x.number)
                except httpx.HTTPError:
                    card = None
                if card and card_names_match(x.card_name, card.get("name") or ""):
                    x.set_code = set_id
                    price = TCGdex.price_entry(card)
                    if price["variants"]:
                        self.prices.setdefault(x.card_key, price)
                    fixed.append(x.listing_id)
                    break
            else:
                if named_wrong:
                    bad.append(x.listing_id)
                    x.set_code = "unknown"
                    x.confidence = min(x.confidence, self.mismatch_conf)
        return fixed, bad

    def ensure_prices(self, listings: list[CardListing], pause_s: float = 0.2) -> list[str]:
        """Fetch prices for cards not cached yet. Returns card keys with no TCGplayer price."""
        missing = []
        for key in sorted({x.card_key for x in listings if x.card_key}):
            if key in self.prices:
                continue
            set_id, _, number = key.rpartition("-")
            try:
                card = self.client.card(set_id, number)
            except httpx.HTTPError:
                card = None
            entry = TCGdex.price_entry(card) if card else None
            if entry and entry["variants"]:
                self.prices[key] = entry
            else:
                missing.append(key)
            time.sleep(pause_s)  # be polite to a free API
        return missing

    def save(self) -> None:
        out = {k: v for k, v in self.prices.items() if k != "_meta"}
        out["_meta"] = {
            "source": "TCGdex (TCGplayer market prices)",
            "fetched": time.strftime("%Y-%m-%d"),
        }
        self.prices_path.parent.mkdir(parents=True, exist_ok=True)
        self.prices_path.write_text(json.dumps(out, indent=1, ensure_ascii=False), encoding="utf-8")

## Run it all

In [ ]:
#| export
def save_listings(listings: list[CardListing], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [x.model_dump() for x in listings]
    path.write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding="utf-8")

In [ ]:
#| export
def run_ingest(
    config: dict[str, Any],
    llm: LLM,
    raw_items: list[dict],
    pricebook: PriceBook | None = None,
    out_path: Path | None = None,
    log=print,
) -> dict[str, Any]:
    """Raw Apify items -> data/listings.json (+ prices, + review sheet). Returns a summary."""
    rows = [r for r in (anonymize(i, config) for i in raw_items) if r]
    log(f"{len(raw_items)} raw items -> {len(rows)} usable listings")
    kept, dropped, stats = identify_listings(config, llm, rows, log=log)
    log(f"in scope: {len(kept)}, dropped: {dropped} (before the LLM: {stats['prefiltered']}), "
        f"{stats['from_cache']} identified earlier, {stats['llm_calls']} LLM calls")  # fmt: skip
    summary: dict[str, Any] = {
        "raw": len(raw_items),
        "usable": len(rows),
        "kept": len(kept),
        "dropped": dropped,
        "identify": stats,
    }
    mismatched: list[str] = []
    if pricebook is not None:
        summary["unresolved_sets"] = pricebook.resolve_sets(kept)
        summary["unpriced"] = pricebook.ensure_prices(kept)
        fixed, mismatched = pricebook.verify_cards(kept)
        summary["set_fixed"], summary["name_mismatch"] = fixed, mismatched
        pricebook.save()
        log(f"prices: {len(summary['unpriced'])} cards without a TCGplayer price, "
            f"sets fixed by TCGdex {len(fixed)}, names matching no card {len(mismatched)}")  # fmt: skip
    save_listings(kept, out_path or repo_path(config["data"]["listings"]))
    min_conf = config["thresholds"]["extraction_confidence_review"]
    prices = pricebook.prices if pricebook is not None else {}
    reasons = review_reasons(kept, prices, mismatched, min_conf)
    write_review_sheet(kept, config, reasons)
    summary["needs_review"] = list(reasons)
    log(f"rows needing a hand check: {len(reasons)}")
    return summary

## Load the listing pool

What the Merchant Agent searches: the samples until `data.use_samples` is false.

In [ ]:
#| export
@dataclass
class MarketData:
    """Everything the Merchant Agent searches and checks against."""

    listings: list[CardListing]
    prices: dict[str, dict]
    red_flags: list[RedFlag]
    source: str  # "samples", "snapshot" or "live"
    meta: dict = field(default_factory=dict)

In [ ]:
#| export
def real_prices(config: dict[str, Any]) -> dict[str, dict]:
    """data/prices.json (TCGdex market prices), never the sample prices."""
    p = repo_path(config["data"]["prices"])
    prices = json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}
    prices.pop("_meta", None)
    return prices

In [ ]:
#| export
def _read_listings(p: Path) -> list[CardListing]:
    return [CardListing.model_validate(x) for x in json.loads(p.read_text(encoding="utf-8"))]

In [ ]:
#| export
def load_data(config: dict[str, Any]) -> MarketData:
    """The sample pool, or the real snapshot once data/listings.json exists and
    `data.use_samples` is false. Live search results are added to the snapshot, never to
    the samples."""
    d = config["data"]
    red_flags = parse_red_flags(repo_path(d["red_flags"]).read_text(encoding="utf-8"))
    use_samples = d["use_samples"] or not repo_path(d["listings"]).exists()
    lp = repo_path(d["sample_listings"] if use_samples else d["listings"])
    pp = repo_path(d["sample_prices"] if use_samples else d["prices"])
    pool = {x.listing_id: x for x in _read_listings(lp)}
    live_path = repo_path(config["merchant"]["live_listings"])
    if not use_samples and live_path.exists():
        pool.update({x.listing_id: x for x in _read_listings(live_path)})
    prices = json.loads(pp.read_text(encoding="utf-8")) if pp.exists() else {}
    meta = prices.pop("_meta", {})
    return MarketData(
        list(pool.values()), prices, red_flags, "samples" if use_samples else "snapshot", meta
    )

## Offline mode

Rule-based stand-ins used when `LIVE = False` (and by the tests).

In [ ]:
#| export
CARD_NAMES = [
    "Charizard ex", "Charizard VMAX", "Charizard V", "Charizard", "Blastoise", "Venusaur",
    "Pikachu VMAX", "Pikachu V", "Pikachu", "Mewtwo", "Zapdos", "Umbreon VMAX", "Umbreon",
    "Gengar", "Mew", "Rayquaza", "Lugia",
]  # fmt: skip
SET_HINTS = [
    (r"\bbase(?: set)?\b", "Base Set"),
    (r"\b151\b", "151"),
    (r"vivid voltage", "Vivid Voltage"),
    (r"evolving skies", "Evolving Skies"),
]
DENOMINATOR_SET = {"102": "Base Set", "165": "151", "185": "Vivid Voltage", "203": "Evolving Skies"}
CONDITION_PATTERNS = [
    (r"\b(nm|near mint|mint)\b", "NM"),
    (r"\b(lp|lightly played|light play)\b", "LP"),
    (r"\b(mp|moderately played)\b", "MP"),
    (r"\b(hp|heavily played|well loved|played)\b", "HP"),
    (r"\b(dmg|damaged|crease|creased|bent|water damage)\b", "DMG"),
]

In [ ]:
#| export
def set_display_name(code: str | None, config: dict[str, Any]) -> str | None:
    """A set id as people say it ("base1" -> "Base Set"), from the cached TCGdex set list;
    anything else comes back unchanged."""
    if not code:
        return code
    path = repo_path(config["data"]["tcgdex_sets"])
    if path.exists():
        for s in json.loads(path.read_text(encoding="utf-8")):
            if s.get("id", "").casefold() == code.casefold():
                return s.get("name") or code
    return code

In [ ]:
#| export
def between(prompt: str, tag: str) -> str:
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", prompt, flags=re.DOTALL)
    return m.group(1) if m else prompt

In [ ]:
#| export
def guess_card_name(text: str) -> str | None:
    low = text.lower()
    return next((n for n in CARD_NAMES if re.search(rf"\b{re.escape(n.lower())}\b", low)), None)

In [ ]:
#| export
def guess_condition(text: str) -> str | None:
    low = text.lower()
    return next((c for p, c in CONDITION_PATTERNS if re.search(p, low)), None)

In [ ]:
#| export
@offline_handler("extract")
def offline_extract(_system: str, prompt: str) -> dict:
    """Rule-based stand-in for the extraction call (offline mode)."""
    cards = []
    for it in json.loads(between(prompt, "listings")):
        text = f"{it.get('title', '')} {it.get('description') or ''}"
        low = text.lower()
        num = re.search(r"\b(\d{1,3})\s*/\s*(\d{2,3})\b", text)
        graded = re.search(r"\b(psa|bgs|cgc)\s*(\d+(?:\.\d)?)", low)
        variant = next((v for k, v in (("reverse", "reverse"), ("1st edition", "1st edition"),
                                       ("holo", "holo")) if k in low), None)  # fmt: skip
        set_name = DENOMINATOR_SET.get(num.group(2)) if num else None
        cards.append({
            "listing_id": it["listing_id"],
            "card_name": guess_card_name(text),
            "set_name": set_name or next((n for p, n in SET_HINTS if re.search(p, low)), None),
            "number": num.group(1).lstrip("0") if num else None,
            "variant": variant,
            "condition": guess_condition(text) or "unknown",
            "graded": bool(graded),
            "grade": graded.group(2) if graded else None,
            "language": "ja" if re.search(r"\b(japanese|jp|jpn)\b", low) else "en",
            "is_lot": bool(re.search(r"\b(lot|bundle|bulk|\d+\s*cards)\b|\+", it.get("title", "").lower())),
            "confidence": 0.9 if num else 0.6,
        })  # fmt: skip
    return {"cards": cards}

# Run it

Offline (default) this runs the whole pipeline on the 28 synthetic raw listings in
`data/samples/sample_raw_listings.json`, writing to a temp folder so nothing real is touched.

To build the real snapshot: set `LIVE = True` (Gemini) and `SCRAPE = True` (Apify, costs
credit; capped by `scrape.max_charge_usd`). It runs one Apify search per entry in
`scrape.queries`, `scrape.per_query` listings each (10 × 10 = 100). `TEST_RUN = True` takes
1 listing per search instead. Then hand-check `data/listings_review.csv` and set
`data.use_samples: false` in config.yaml. To redo identification without scraping again,
load the newest raw file with `latest_raw(cfg)`.

In [ ]:
import copy
import json
import tempfile

import pandas as pd

from src.config import load_config, repo_path
from src.llm import make_llm

SCRAPE = False  # True: run the Apify actor (needs APIFY_API_TOKEN, costs credit)
TEST_RUN = True  # 1 listing per search instead of scrape.per_query

cfg = load_config()
llm = make_llm(cfg)
if not SCRAPE:  # dry run: sample input, temp outputs
    tmp = Path(tempfile.mkdtemp())
    cfg = copy.deepcopy(cfg)
    for key in ("listings", "prices", "review_sheet", "tcgdex_sets"):
        cfg["data"][key] = str(tmp / Path(cfg["data"][key]).name)
    print("dry run, writing to", tmp)

In [ ]:
if SCRAPE:
    raw = scrape_snapshot(cfg, MarketplaceScraper(cfg), per_query=1 if TEST_RUN else None)
else:
    raw = json.loads(repo_path(cfg["data"]["sample_raw"]).read_text(encoding="utf-8"))
print(len(raw), "raw items; first one anonymized:")
anonymize(raw[0])

Prices: real TCGdex calls when scraping; in the dry run, the sample price file stands in.

In [ ]:
if SCRAPE:
    book = PriceBook(cfg, {})
else:
    sample_prices = json.loads(repo_path(cfg["data"]["sample_prices"]).read_text(encoding="utf-8"))
    sample_prices.pop("_meta", None)

    class SampleTCGdex(TCGdex):  # answers from the sample prices, no network
        def __init__(self):
            pass

        def sets(self):
            return [
                {"id": "base1", "name": "Base Set"},
                {"id": "sv03.5", "name": "151"},
                {"id": "swsh4", "name": "Vivid Voltage"},
            ]

        def card(self, set_id, number):
            e = sample_prices.get(f"{set_id}-{number}")
            return e and {
                "name": e["name"],
                "set": {"name": e["set_name"]},
                "pricing": {"tcgplayer": {k: {"marketPrice": v} for k, v in e["variants"].items()}},
            }

    book = PriceBook(cfg, {}, client=SampleTCGdex())

summary = run_ingest(cfg, llm, raw, pricebook=book)
summary

In [ ]:
listings = json.loads(repo_path(cfg["data"]["listings"]).read_text(encoding="utf-8"))
cols = [
    "listing_id",
    "raw_title",
    "price",
    "card_name",
    "set_name",
    "set_code",
    "number",
    "condition",
    "confidence",
]
pd.DataFrame(listings)[cols]

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()